# ClickRemoval Test Tutorial

This notebook follows the simple tutorial style of `LLaVA_OneVision_Tutorials.ipynb`: set up the environment, load one model configuration, prepare image + click coordinates, then run inference.

It is built from `test-1x.py`, `test-2x.py`, and `test-xl.py` without changing those original scripts.

## 1. Setup

Update `PROJECT_ROOT` if your ClickRemoval folder is in another location.

In [ ]:
from pathlib import Path
import gc
import json
import os
import sys
import traceback
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start_path=None):
    if start_path is None:
        start_path = Path.cwd()
    for parent in [start_path] + list(start_path.parents):
        if (parent / "examples").exists():
            return parent
    return start_path

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {Path.cwd()}")
print(f"examples directory exists: {(PROJECT_ROOT / 'examples').exists()}")

## 2. Imports

These imports are collected from the three test files. The XL-only unused image utility imports were omitted because the current test script does not use them during inference.

In [ ]:
import torch
from diffusers import DDIMScheduler, DiffusionPipeline
from IPython.display import display

from src.stable_diffusion_1_attention_aggregator import StableDiffusion1AttentionAggregator
from src.stable_diffusion_2_attention_aggregator import StableDiffusion2AttentionAggregator
from src.stable_diffusion_xl_attention_aggregator import StableDiffusionxlAttentionAggregator

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
aggregator_device = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Device: {device}")
print(f"dtype: {dtype}")

## 3. Model Configurations

Choose one version:

- `sd15`: from `test-1x.py`
- `sd21`: from `test-2x.py`
- `sdxl`: from `test-xl.py`

In [ ]:
MODEL_CONFIGS = {
    "sd15": {
        "display_name": "Stable Diffusion v1.5",
        "model_path": "./models/stable-diffusion-v1-5",
        "custom_pipeline": "./pipelines/pipline1x.py",
        "aggregator_cls": StableDiffusion1AttentionAggregator,
        "aggregator_kwargs": "pipe",
        "height": 512,
        "width": 512,
        "rm_guidance_scale": 6,
        "sg_steps": 5,
        "sg_scale": 0.3,
        "SGA_start_step": 0,
        "SGA_start_layer": 7,
        "SGA_end_layer": 16,
        "num_inference_steps": 25,
    },
    "sd21": {
        "display_name": "Stable Diffusion 2.1 Base",
        "model_path": "./models/stable-diffusion-2-1-base",
        "custom_pipeline": "./pipelines/pipline2x.py",
        "aggregator_cls": StableDiffusion2AttentionAggregator,
        "aggregator_kwargs": "pipe",
        "height": 512,
        "width": 512,
        "rm_guidance_scale": 6,
        "sg_steps": 5,
        "sg_scale": None,
        "SGA_start_step": 0,
        "SGA_start_layer": 7,
        "SGA_end_layer": 16,
        "num_inference_steps": 50,
    },
    "sdxl": {
        "display_name": "Stable Diffusion XL Base 1.0",
        "model_path": "./models/stable-diffusion-xl-base-1.0",
        "custom_pipeline": "./pipelines/piplinexl.py",
        "aggregator_cls": StableDiffusionxlAttentionAggregator,
        "aggregator_kwargs": "device_only",
        "height": 1024,
        "width": 1024,
        "rm_guidance_scale": 7,
        "sg_steps": 9,
        "sg_scale": 0.3,
        "SGA_start_step": 0,
        "SGA_start_layer": 34,
        "SGA_end_layer": 70,
        "num_inference_steps": 50,
    },
}

list(MODEL_CONFIGS)

## 4. Helper Functions

The coordinate loader keeps the same JSON format as the test scripts: `points` and `labels`.

In [ ]:
def load_coordinates(json_path):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        points = data.get("points", [])
        labels = data.get("labels", [])
        formatted_points = []

        for point in points:
            if len(point) == 2:
                formatted_points.append((point[0], point[1]))

        return formatted_points, labels
    except Exception:
        traceback.print_exc()
        return [], []


def make_scheduler():
    return DDIMScheduler(
        beta_start=0.00085,
        beta_end=0.012,
        beta_schedule="scaled_linear",
        clip_sample=False,
        set_alpha_to_one=False,
    )


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def check_input_files(image_path, json_path):
    image_path = Path(image_path)
    json_path = Path(json_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")
    if not json_path.exists():
        raise FileNotFoundError(f"JSON not found: {json_path}")

    points, labels = load_coordinates(json_path)
    if not points:
        raise ValueError(f"No valid points found in: {json_path}")
    if len(points) != len(labels):
        raise ValueError(f"points/labels length mismatch: {len(points)} vs {len(labels)}")

    print(f"Image: {image_path}")
    print(f"JSON: {json_path}")
    print(f"Loaded {len(points)} points and {len(labels)} labels")
    return points, labels

## 5. Load a Model

This mirrors the model-loading part of the three scripts, but wraps it behind `load_click_removal_model(version)`.

In [ ]:
def load_click_removal_model(version="sd15"):
    if version not in MODEL_CONFIGS:
        raise ValueError(f"Unknown version: {version}. Choose from {list(MODEL_CONFIGS)}")

    cfg = MODEL_CONFIGS[version]
    print(f"Loading {cfg['display_name']}...")

    pipe = DiffusionPipeline.from_pretrained(
        cfg["model_path"],
        custom_pipeline=cfg["custom_pipeline"],
        scheduler=make_scheduler(),
        variant="fp16" if torch.cuda.is_available() else None,
        use_safetensors=True,
        torch_dtype=dtype,
    ).to(device)

    pipe.enable_attention_slicing()
    if torch.cuda.is_available():
        pipe.enable_model_cpu_offload()

    if cfg["aggregator_kwargs"] == "pipe":
        attn_aggregator = cfg["aggregator_cls"](pipe=pipe, device=aggregator_device)
    else:
        attn_aggregator = cfg["aggregator_cls"](device=aggregator_device)

    clear_memory()
    print("Model ready.")
    return pipe, attn_aggregator, cfg

## 6. Run Inference

The default input is the same as the test scripts: `./examples/test1.jpg` and `./examples/test1.json`.

In [ ]:
def run_click_removal(
    version="sd15",
    image_path="./examples/test1.jpg",
    json_path="./examples/test1.json",
    output_dir="./out_notebook",
    prompt="",
    seed=123,
):
    points, labels = check_input_files(image_path, json_path)
    pipe, attn_aggregator, cfg = load_click_removal_model(version)

    generator = torch.Generator(device=device).manual_seed(seed)

    result = pipe(
        prompt=prompt,
        image=image_path,
        points=points,
        points_in_segment=labels,
        height=cfg["height"],
        width=cfg["width"],
        SGA=True,
        strength=0.8,
        rm_guidance_scale=cfg["rm_guidance_scale"],
        sg_steps=cfg["sg_steps"],
        sg_scale=cfg["sg_scale"],
        SGA_start_step=cfg["SGA_start_step"],
        SGA_start_layer=cfg["SGA_start_layer"],
        SGA_end_layer=cfg["SGA_end_layer"],
        num_inference_steps=cfg["num_inference_steps"],
        generator=generator,
        guidance_scale=1,
        attn_aggregator=attn_aggregator,
    )

    image = result.images[0]

    from PIL import Image as PILImage
    with PILImage.open(image_path) as orig_img:
        orig_w, orig_h = orig_img.size
    target_size = cfg["height"]
    scale = target_size / max(orig_w, orig_h)
    new_w = int(orig_w * scale)
    new_h = int(orig_h * scale)
    pad_left = (target_size - new_w) // 2
    pad_top = (target_size - new_h) // 2
    if pad_left > 0 or pad_top > 0:
        image = image.crop((pad_left, pad_top, pad_left + new_w, pad_top + new_h))

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{Path(image_path).stem}_{version}.png"
    image.save(output_path)

    print(f"Saved result to: {output_path}")
    display(image)

    del pipe, attn_aggregator
    clear_memory()
    return image, output_path

## 7. Example: Stable Diffusion v1.5

This corresponds to `test-1x.py`.

In [ ]:
# image, output_path = run_click_removal(version="sd15")

## 8. Example: Stable Diffusion 2.1

This corresponds to `test-2x.py`.

In [ ]:
# image, output_path = run_click_removal(version="sd21")

## 9. Example: Stable Diffusion XL

This corresponds to `test-xl.py`. It uses 1024 x 1024 output and usually needs more GPU memory.

In [ ]:
# image, output_path = run_click_removal(version="sdxl")

## 10. Custom Input

Use this cell when you want to test another image and click JSON file.

In [ ]:
# image, output_path = run_click_removal(
#     version="sd15",
#     image_path="./examples/test1.jpg",
#     json_path="./examples/test1.json",
#     output_dir="./out_notebook",
#     seed=123,
# )

## 11. Run All Versions

Use this only when you have enough GPU memory and time. For most experiments, running one version at a time is easier to debug.

In [ ]:
# outputs = {}
# for version in ["sd15", "sd21", "sdxl"]:
#     print(f"\n===== Running {version} =====")
#     _, output_path = run_click_removal(version=version)
#     outputs[version] = output_path
# outputs